# Kasus 38 — Marine Oil Spill Segmentation (Sentinel-1 SAR) dengan U-Net Ringan
Pipeline: audit data → split reproducible → U-Net ringan (GPU T4) → evaluasi test → ONNX FP32/FP16/INT8 → artefak web (onnxruntime-web).

**Runtime:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Jalankan sel berurutan dari atas.

## 0. Cek GPU

In [ ]:
# === 0. Cek runtime GPU ===
import platform, torch, subprocess
print("Python", platform.python_version(), "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU tidak aktif! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"], capture_output=True, text=True).stdout)

# Helper: cetak traceback juga sebagai teks biasa (stdout) agar mudah dibaca/di-log
import traceback
def _print_exc(shell, etype, evalue, tb, tb_offset=None):
    print("TRACEBACK:\n" + "".join(traceback.format_exception(etype, evalue, tb))[-3000:])
    return shell.InteractiveTB.structured_traceback(etype, evalue, tb, tb_offset=tb_offset)
get_ipython().set_custom_exc((Exception,), _print_exc)

## 1. Dataset
Dataset publik Kaggle `bitsandlayers/sar-oil-spill-segmentation-dataset-sos` (lisensi di halaman Kaggle: CC BY 4.0). Diunduh via `kagglehub` tanpa token.

In [ ]:
# === 1. Download dataset SOS via kagglehub ===
import kagglehub, os
from collections import Counter
path = kagglehub.dataset_download("bitsandlayers/sar-oil-spill-segmentation-dataset-sos")
print("Path to dataset files:", path)
for root, dirs, files in sorted(os.walk(path)):
    depth = root[len(path):].count(os.sep)
    exts = Counter(os.path.splitext(f)[1].lower() for f in files)
    print("  " * depth + (os.path.basename(root) or root) + "/", f"[{len(files)} files]", dict(exts) if exts else "")
    if depth < 5 and files:
        print("  " * (depth + 1) + "contoh:", sorted(files)[:3])

## 2. Audit data
Format, pasangan citra–mask, statistik, duplikat eksak/near-duplicate (termasuk rotasi/flip), dan visualisasi. Semua hasil disimpan di `/content/outputs/audit`.

In [ ]:
# === 2a. Probe cepat format file (beberapa sampel per sensor) ===
for split in ["train", "test"]:
    for sensor in ["palsar", "sentinel"]:
        for stem in ["0", "1", "100"]:
            ip = DATA_ROOT / split / sensor / "image" / f"{stem}.png"; lp = DATA_ROOT / split / sensor / "label" / f"{stem}.png"
            im, lb = Image.open(ip), Image.open(lp); a, m = np.array(im), np.array(lb)
            flat = m.reshape(-1, m.shape[-1]) if m.ndim == 3 else m.reshape(-1, 1)
            code = (flat.astype(np.int64) * (256 ** np.arange(flat.shape[1]))).sum(1)
            vals, cnts = np.unique(code, return_counts=True)
            print(f"{split}/{sensor}/{stem}: img {im.mode} {im.size} {a.dtype} rng[{a.min()},{a.max()}] | lbl {lb.mode} {lb.size} uniq={len(vals)} top={[(flat[code==v][0].tolist(), int(c)) for v, c in sorted(zip(vals, cnts), key=lambda t: -t[1])[:4]]}")

In [ ]:
# === 2b. Audit lengkap: pasangan, dimensi, channel, nilai mask, duplikat ===
import numpy as np, pandas as pd, hashlib, json, time
from concurrent.futures import ThreadPoolExecutor
OUT = Path("/content/outputs"); AUDIT = OUT / "audit"; AUDIT.mkdir(parents=True, exist_ok=True); (OUT / "src").mkdir(exist_ok=True)

def ahash(g, n=16):  # average-hash 16x16 untuk deteksi near-duplicate
    h, w = g.shape; b = g[: h // n * n, : w // n * n].reshape(n, h // n, n, w // n).mean((1, 3))
    return np.packbits((b > b.mean()).ravel()).tobytes().hex()

def inspect_pair(args):
    split, sensor, stem, ip, lp = args
    im, lb = Image.open(ip), Image.open(lp); a, m = np.array(im), np.array(lb)
    a3 = a if a.ndim == 3 else a[..., None]; m3 = m if m.ndim == 3 else m[..., None]
    img_gray = bool((a3 == a3[..., :1]).all()); lbl_gray = bool((m3 == m3[..., :1]).all())
    g, ml = a3[..., 0].astype(np.float32), m3[..., 0]
    hist = np.bincount(ml.ravel(), minlength=256)
    return dict(split=split, sensor=sensor, stem=stem, img_path=str(ip), lbl_path=str(lp),
        img_w=im.size[0], img_h=im.size[1], img_mode=im.mode, lbl_w=lb.size[0], lbl_h=lb.size[1], lbl_mode=lb.mode,
        img_channels_equal=img_gray, lbl_channels_equal=lbl_gray,
        img_min=int(a.min()), img_max=int(a.max()), img_mean=float(g.mean()), img_std=float(g.std()),
        img_nonfinite=0 if np.issubdtype(a.dtype, np.integer) else int((~np.isfinite(a)).sum()),
        lbl_n_unique=int((hist > 0).sum()), lbl_px_0=int(hist[0]), lbl_px_255=int(hist[255]),
        lbl_px_mid=int(hist[1:255].sum()), lbl_px_ge128=int(hist[128:].sum()),
        fg_frac=float(hist[128:].sum() / ml.size),
        img_md5=hashlib.md5(a.tobytes()).hexdigest(), lbl_md5=hashlib.md5(m.tobytes()).hexdigest(),
        img_ahash=ahash(g), img_bytes=os.path.getsize(ip), lbl_bytes=os.path.getsize(lp))

tasks, pair_report = [], []
for split in ["train", "test"]:
    for sensor in sorted(p.name for p in (DATA_ROOT / split).iterdir() if p.is_dir()):
        idir, ldir = DATA_ROOT / split / sensor / "image", DATA_ROOT / split / sensor / "label"
        imgs = {p.stem: p for p in idir.iterdir() if p.is_file()}
        lbls = {p.stem: p for p in ldir.iterdir() if p.is_file()}
        both = sorted(set(imgs) & set(lbls), key=lambda s: (len(s), s))
        pair_report.append(dict(split=split, sensor=sensor, n_img=len(imgs), n_lbl=len(lbls), n_pairs=len(both),
            img_without_lbl=len(set(imgs) - set(lbls)), lbl_without_img=len(set(lbls) - set(imgs)),
            examples_unpaired=json.dumps(sorted(set(imgs) ^ set(lbls))[:20])))
        tasks += [(split, sensor, s, imgs[s], lbls[s]) for s in both]

t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    df = pd.DataFrame(list(ex.map(inspect_pair, tasks, chunksize=64)))
print(f"Diinspeksi {len(df)} pasangan dalam {time.time()-t0:.1f}s")
df.to_csv(AUDIT / "audit_per_file.csv", index=False)
pr = pd.DataFrame(pair_report); pr.to_csv(AUDIT / "pairing_report.csv", index=False)
print(pr[["split", "sensor", "n_img", "n_lbl", "n_pairs", "img_without_lbl", "lbl_without_img"]].to_string(index=False))

In [ ]:
# === 2c. Ringkasan statistik audit + duplikasi/kebocoran ===
rows = []
for (split, sensor), d in df.groupby(["split", "sensor"]):
    npx = (d.lbl_w * d.lbl_h).sum()
    rows.append(dict(split=split, sensor=sensor, n=len(d),
        sizes=str(sorted(set(zip(d.img_w, d.img_h)))), size_mismatch=int(((d.img_w != d.lbl_w) | (d.img_h != d.lbl_h)).sum()),
        img_modes=str(dict(d.img_mode.value_counts())), lbl_modes=str(dict(d.lbl_mode.value_counts())),
        img_ch_equal_pct=100 * d.img_channels_equal.mean(), lbl_ch_equal_pct=100 * d.lbl_channels_equal.mean(),
        lbl_mid_px_pct=100 * d.lbl_px_mid.sum() / npx, files_w_mid_px=int((d.lbl_px_mid > 0).sum()),
        fg_px_pct=100 * d.lbl_px_ge128.sum() / npx, fg_median_pct=100 * d.fg_frac.median(),
        empty_mask=int((d.fg_frac == 0).sum()), full_mask=int((d.fg_frac == 1).sum()),
        img_mean=d.img_mean.mean(), img_std=d.img_std.mean(), const_img=int((d.img_min == d.img_max).sum()),
        nonfinite=int(d.img_nonfinite.sum())))
summary = pd.DataFrame(rows).round(3)
summary.to_csv(AUDIT / "summary_by_split_sensor.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(summary.drop(columns=["sizes", "img_modes", "lbl_modes"]).T.to_string(header=False))
    print(summary[["split", "sensor", "sizes", "img_modes", "lbl_modes"]].to_string(index=False))

# Duplikasi eksak (md5 piksel) & near-duplicate (average-hash) dalam dan antar split, per sensor
dup = {}
for key in ["img_md5", "img_ahash"]:
    for sensor in sorted(df.sensor.unique()):
        d = df[df.sensor == sensor]; tr, te = d[d.split == "train"], d[d.split == "test"]
        dup[f"{sensor}/{key}/within_train_dupe_rows"] = int(tr[key].duplicated(keep=False).sum())
        dup[f"{sensor}/{key}/within_test_dupe_rows"] = int(te[key].duplicated(keep=False).sum())
        dup[f"{sensor}/{key}/test_rows_also_in_train"] = int(te[key].isin(set(tr[key])).sum())
    dup[f"cross_sensor/{key}/shared_hashes"] = len(set(df[df.sensor == "palsar"][key]) & set(df[df.sensor == "sentinel"][key]))
print(json.dumps(dup, indent=1))
json.dump(dup, open(AUDIT / "duplicates.json", "w"), indent=1)

In [ ]:
# === 2d. Near-duplicate termasuk rotasi/flip (8 transformasi dihedral), jarak Hamming aHash ===
def grid16(p):
    g = np.asarray(Image.open(p), dtype=np.float32)
    g = g[..., 0] if g.ndim == 3 else g
    return g.reshape(16, 16, 16, 16).mean((1, 3))

def bits(b): return np.packbits((b > b.mean()).ravel())
def dihedral(b): return [np.rot90(b, k) for k in range(4)] + [np.rot90(b.T, k) for k in range(4)]

with ThreadPoolExecutor(4) as ex:
    grids = list(ex.map(grid16, df.img_path, chunksize=64))
H0 = np.stack([bits(g) for g in grids])                                  # (N, 32) hash asli
HD = np.stack([np.stack([bits(t) for t in dihedral(g)]) for g in grids])  # (N, 8, 32) semua varian
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def min_hamming(q_idx, ref_idx):  # jarak min varian query vs hash asli referensi
    ref = H0[ref_idx]; out = np.full(len(q_idx), 999); arg = np.full(len(q_idx), -1)
    for s in range(0, len(q_idx), 256):
        q = HD[q_idx[s:s+256]]                                              # (b, 8, 32)
        d = POP[np.bitwise_xor(q[:, :, None, :], ref[None, None])].sum(-1).min(1)  # (b, R)
        out[s:s+256], arg[s:s+256] = d.min(1), d.argmin(1)
    return out, arg

TH = 8  # <= 8 dari 256 bit (~3%) dianggap near-duplicate
neardup = {}
for sensor in sorted(df.sensor.unique()):
    tr = np.where((df.sensor == sensor) & (df.split == "train"))[0]; te = np.where((df.sensor == sensor) & (df.split == "test"))[0]
    d_te, _ = min_hamming(te, tr)
    # dalam train: bandingkan dengan semua train lain (abaikan diri sendiri)
    d_tr = np.full(len(tr), 999)
    for s in range(0, len(tr), 256):
        q = HD[tr[s:s+256]]; d = POP[np.bitwise_xor(q[:, :, None, :], H0[tr][None, None])].sum(-1).min(1)
        d[np.arange(len(q)), np.arange(s, s + len(q))] = 999; d_tr[s:s+256] = d.min(1)
    neardup[sensor] = dict(test_near_train=int((d_te <= TH).sum()), train_near_train=int((d_tr <= TH).sum()),
        test_min_dist_p1=float(np.percentile(d_te, 1)), train_min_dist_p1=float(np.percentile(d_tr, 1)))
    df.loc[te, "nn_dist_to_train"] = d_te; df.loc[tr, "nn_dist_within_train"] = d_tr
print(json.dumps(neardup, indent=1))
json.dump(neardup, open(AUDIT / "near_duplicates_dihedral.json", "w"), indent=1)
df.to_csv(AUDIT / "audit_per_file.csv", index=False)

In [ ]:
# === 2e. Visualisasi: contoh pasangan citra-mask-overlay & pasangan test~train terdekat ===
import matplotlib.pyplot as plt
def load_gray(p): a = np.asarray(Image.open(p)); return a[..., 0] if a.ndim == 3 else a
def overlay(img, m):
    rgb = np.stack([img] * 3, -1).astype(np.float32) / 255; fg = m >= 128
    rgb[fg] = 0.55 * rgb[fg] + 0.45 * np.array([1.0, 0.15, 0.1]); return rgb

rng = np.random.default_rng(0)
fig, axes = plt.subplots(4, 6, figsize=(15, 10.5))
for r, (split, sensor) in enumerate([("train", "palsar"), ("train", "sentinel"), ("test", "palsar"), ("test", "sentinel")]):
    d = df[(df.split == split) & (df.sensor == sensor)]
    for k, i in enumerate(rng.choice(d.index, 2, replace=False)):
        img, m = load_gray(df.img_path[i]), load_gray(df.lbl_path[i])
        for j, (x, t) in enumerate([(img, "image"), (m, "mask"), (overlay(img, m), "overlay")]):
            ax = axes[r, 3 * k + j]; ax.imshow(x, cmap="gray" if x.ndim == 2 else None, vmin=0, vmax=255 if x.ndim == 2 else None)
            ax.set_title(f"{split}/{sensor}/{df.stem[i]} {t}" + (f" fg={df.fg_frac[i]:.2f}" if j == 1 else ""), fontsize=8); ax.axis("off")
plt.tight_layout(); plt.savefig(AUDIT / "sample_pairs_grid.png", dpi=110); plt.show()

# 6 pasangan test->train terdekat (sentinel & palsar) untuk cek near-duplicate secara visual
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for r, sensor in enumerate(["sentinel", "palsar"]):
    te = df[(df.sensor == sensor) & (df.split == "test")].sort_values("nn_dist_to_train").head(3)
    tr_idx = np.where((df.sensor == sensor) & (df.split == "train"))[0]
    _, arg = min_hamming(te.index.values, tr_idx)
    for k, (i, a) in enumerate(zip(te.index, arg)):
        j = tr_idx[a]
        axes[r, 2 * k].imshow(load_gray(df.img_path[i]), cmap="gray"); axes[r, 2 * k].set_title(f"TEST {sensor}/{df.stem[i]} d={int(df.nn_dist_to_train[i])}", fontsize=8)
        axes[r, 2 * k + 1].imshow(load_gray(df.img_path[j]), cmap="gray"); axes[r, 2 * k + 1].set_title(f"TRAIN {sensor}/{df.stem[j]}", fontsize=8)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.savefig(AUDIT / "nearest_test_train_pairs.png", dpi=110); plt.show()

## 3. Konfigurasi & split
Sensor utama: **Sentinel-1** (sampel terbanyak, lebih sedikit label ambigu, sesuai katalog). Val 15% dari train dengan grup near-duplicate; test hanya untuk evaluasi final.

In [ ]:
# === 3. Konfigurasi eksperimen & split train/val berbasis grup (reproducible) ===
import random, sys, importlib.metadata as md
CFG = dict(
    sensor="sentinel",   # dipilih dari audit: sampel terbanyak (3354), lebih sedikit file mask ambigu, sesuai katalog (Sentinel-1), data open-access
    seed=42, val_frac=0.15, block_size=1,      # ID file terbukti teracak (lihat cek di bawah) -> grup = komponen near-duplicate saja
    img_size=256,        # ukuran asli 256x256 dipertahankan (tanpa resize)
    in_channels=1,       # 3 channel RGB identik -> pakai 1 channel
    mask_threshold=128,  # label: piksel >= 128 -> oil (1), selain itu 0 (diturunkan dari audit: nilai 0/255 + 2.5% piksel antara di train)
    base_ch=16, epochs=40, batch_size=16, lr=1e-3, weight_decay=1e-4, patience=8, bce_weight=0.5,
)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["seed"])

d_all = df[df.sensor == CFG["sensor"]].copy()
d_all["id"] = d_all.stem.astype(int)
trainpool = d_all[d_all.split == "train"].sort_values("id").reset_index()
test_df = d_all[d_all.split == "test"].sort_values("id").reset_index()

# Cek apakah ID berurutan cenderung mirip (indikasi patch dari scene yang sama)
Hs = H0[trainpool["index"].values]
ham = lambda a, b: POP[np.bitwise_xor(a, b)].sum(-1)
consec = ham(Hs[:-1], Hs[1:]); rnd = ham(Hs, Hs[np.random.default_rng(0).permutation(len(Hs))])
print(f"Hamming aHash ID berurutan: median={np.median(consec):.0f} | pasangan acak: median={np.median(rnd):.0f}")

# Grup: near-duplicate (termasuk rotasi/flip) dalam train digabung ke grup yang sama (union-find) agar tidak terpisah ke train & val
parent = {i: i for i in range(len(trainpool))}
def find(i):
    while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
    return i
blocks = (trainpool.id // CFG["block_size"]).values
for i in range(1, len(trainpool)):
    if blocks[i] == blocks[i - 1]: parent[find(i)] = find(i - 1)
q = HD[trainpool["index"].values]; ref = H0[trainpool["index"].values]; n_links = 0
for s in range(0, len(q), 256):
    dd = POP[np.bitwise_xor(q[s:s+256, :, None, :], ref[None, None])].sum(-1).min(1)
    for a, b in zip(*np.where(dd <= TH)):
        if s + a != b: parent[find(s + a)] = find(b); n_links += 1
trainpool["group"] = [find(i) for i in range(len(trainpool))]
groups = trainpool.group.unique(); rng = np.random.default_rng(CFG["seed"]); rng.shuffle(groups)
val_groups, n = set(), 0
for g_ in groups:
    if n >= CFG["val_frac"] * len(trainpool): break
    val_groups.add(g_); n += (trainpool.group == g_).sum()
trainpool["subset"] = np.where(trainpool.group.isin(val_groups), "val", "train")
tr_df, va_df = trainpool[trainpool.subset == "train"], trainpool[trainpool.subset == "val"]
assert not set(tr_df.group) & set(va_df.group)
print(f"grup={len(groups)} | link near-dup={n_links} | train={len(tr_df)} val={len(va_df)} test={len(test_df)}")
print(f"fg% train={100*tr_df.fg_frac.mean():.1f} val={100*va_df.fg_frac.mean():.1f} test={100*test_df.fg_frac.mean():.1f}")
SPLIT_DIR = OUT / "splits"; SPLIT_DIR.mkdir(exist_ok=True)
pd.concat([trainpool.assign(subset=trainpool.subset), test_df.assign(subset="test", group=-1)])[["stem", "subset", "group", "img_path", "lbl_path", "fg_frac"]].to_csv(SPLIT_DIR / f"split_{CFG['sensor']}.csv", index=False)
VERSIONS = {p: md.version(p) for p in ["torch", "numpy", "pandas", "pillow", "kagglehub", "matplotlib"]}
VERSIONS["python"] = sys.version.split()[0]; print(VERSIONS)

## 4. Preprocessing
Channel 0 (R=G=B), 256×256 tanpa resize, mask ≥128 → 1, normalisasi mean/std dari subset train.

In [ ]:
# === 4. Preprocessing: load ke memori, binarisasi mask, normalisasi dari statistik TRAIN saja ===
import torch, torch.nn as nn, torch.nn.functional as F
DEV = torch.device("cuda")

def load_set(d):
    X = np.stack([load_gray(p) for p in d.img_path])                                    # (N,256,256) uint8, channel 0
    Y = np.stack([(load_gray(p) >= CFG["mask_threshold"]).astype(np.uint8) for p in d.lbl_path])
    assert X.shape[1:] == (CFG["img_size"], CFG["img_size"]) and X.shape == Y.shape
    return X, Y

t0 = time.time()
Xtr, Ytr = load_set(tr_df); Xva, Yva = load_set(va_df); Xte, Yte = load_set(test_df)
print(f"load {time.time()-t0:.1f}s | train {Xtr.shape} val {Xva.shape} test {Xte.shape}")

# Statistik normalisasi (skala 0-1) hanya dari subset train -> dipakai identik untuk val/test/inferensi web
MEAN = float(Xtr.mean() / 255.0); STD = float(Xtr.std() / 255.0)
PREPROC = dict(input_name="image", input_shape=[1, 1, CFG["img_size"], CFG["img_size"]], layout="NCHW", dtype="float32",
    channel_source="R channel of RGB PNG (R=G=B in dataset) or grayscale", scale="x/255", mean=round(MEAN, 6), std=round(STD, 6),
    formula="(x/255 - mean) / std", resize="none if 256x256; otherwise bilinear resize to 256x256 (mask: nearest)",
    output="logits (1,1,256,256) -> sigmoid -> prob; mask = prob >= threshold", mask_label_rule=f"label pixel >= {CFG['mask_threshold']} -> 1 (oil)")
json.dump(PREPROC, open(OUT / "preprocessing_config.json", "w"), indent=1)
print(f"MEAN={MEAN:.4f} STD={STD:.4f}")

to_gpu = lambda a: torch.from_numpy(a).to(DEV)
gXtr, gYtr, gXva, gYva, gXte, gYte = map(to_gpu, [Xtr, Ytr, Xva, Yva, Xte, Yte])
def prep(xb): return ((xb.float() / 255.0) - MEAN) / STD          # uint8 (B,H,W) -> float (B,1,H,W)
print(f"GPU mem terpakai: {torch.cuda.memory_allocated()/1e6:.0f} MB")

## 5. Metrik & baseline
Baseline: mask kosong dan dark-spot thresholding (ambang dipilih di train).

In [ ]:
# === 5. Metrik & baseline ===
def prep(xb): return (((xb.float() / 255.0) - MEAN) / STD).unsqueeze(1)   # uint8 (B,H,W) -> float (B,1,H,W)

@torch.no_grad()
def seg_metrics(pred, gt):
    """pred, gt: bool/uint8 tensor (N,H,W). Global (agregasi piksel) + macro per-citra.
    Konvensi per-citra: GT & prediksi sama-sama kosong -> Dice=IoU=1."""
    p, g = pred.bool().flatten(1), gt.bool().flatten(1)
    tp = (p & g).sum(1).double(); fp = (p & ~g).sum(1).double(); fn = (~p & g).sum(1).double(); tn = (~p & ~g).sum(1).double()
    TP, FP, FN, TN = tp.sum().item(), fp.sum().item(), fn.sum().item(), tn.sum().item()
    eps = 1e-12
    empty = (tp + fp + fn) == 0
    dice_i = torch.where(empty, 1.0, 2 * tp / (2 * tp + fp + fn + eps)); iou_i = torch.where(empty, 1.0, tp / (tp + fp + fn + eps))
    has_fg = g.any(1)
    return dict(n_images=int(p.shape[0]), fg_pixel_frac=(TP + FN) / (TP + FP + FN + TN),
        TP=int(TP), FP=int(FP), FN=int(FN), TN=int(TN),
        dice_global=2 * TP / (2 * TP + FP + FN + eps), iou_global=TP / (TP + FP + FN + eps),
        precision_global=TP / (TP + FP + eps), recall_global=TP / (TP + FN + eps), accuracy_global=(TP + TN) / (TP + FP + FN + TN),
        dice_macro=dice_i.mean().item(), iou_macro=iou_i.mean().item(),
        dice_macro_fg_only=dice_i[has_fg].mean().item(), n_images_empty_gt=int((~has_fg).sum())), dice_i.cpu().numpy()

@torch.no_grad()
def pr_auc(prob, gt, bins=1000):
    """PR-AUC (average precision) piksel via histogram skor (bins=1000) -> hemat memori di GPU."""
    idx = (prob.flatten().float().clamp(0, 1) * (bins - 1)).long(); gtf = gt.flatten().bool()
    pos = torch.bincount(idx[gtf], minlength=bins).flip(0).cumsum(0).double(); neg = torch.bincount(idx[~gtf], minlength=bins).flip(0).cumsum(0).double()
    prec = pos / (pos + neg).clamp(min=1); rec = pos / pos[-1]
    return float(((rec[1:] - rec[:-1]) * prec[1:]).sum() + rec[0] * prec[0])

# Baseline 1: selalu prediksi kosong (kelas mayoritas "bukan minyak")
BASE = {}
for name, gY in [("val", gYva), ("test", gYte)]:
    BASE[f"empty_mask/{name}"], _ = seg_metrics(torch.zeros_like(gY), gY)

# Baseline 2: dark-spot thresholding klasik (slick minyak tampak gelap di SAR): blur 7x7 lalu intensitas < t.
def dark_spot(gX, t, k=7):
    return F.avg_pool2d(gX.float().unsqueeze(1), k, 1, k // 2).squeeze(1) < t
cands = list(range(20, 161, 5))
sub = torch.arange(0, len(gXtr), 3, device=DEV)   # t dipilih pada subset TRAIN (bukan val/test)
dice_by_t = [seg_metrics(dark_spot(gXtr[sub], t), gYtr[sub])[0]["dice_global"] for t in cands]
T_DARK = cands[int(np.argmax(dice_by_t))]
for name, gX, gY in [("val", gXva, gYva), ("test", gXte, gYte)]:
    BASE[f"dark_threshold_t{T_DARK}/{name}"], _ = seg_metrics(dark_spot(gX, T_DARK), gY)
show = ["dice_global", "iou_global", "precision_global", "recall_global", "accuracy_global", "dice_macro", "iou_macro"]
print(f"T_DARK dipilih di train = {T_DARK}")
print(pd.DataFrame(BASE).T[show].round(4).to_string())

## 6. Model U-Net ringan
Ditulis sebagai modul `src/unet_lite.py` agar dipakai ulang oleh skrip ekspor.

In [ ]:
%%writefile /content/outputs/src/unet_lite.py
# === 6. Model U-Net ringan (modul, dipakai notebook & skrip ekspor ONNX) ===
import torch, torch.nn as nn, torch.nn.functional as F

class DoubleConv(nn.Sequential):
    def __init__(self, i, o):
        super().__init__(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNetLite(nn.Module):
    """U-Net 4 level, kanal base*[1,2,4,8,16]; hanya Conv/BN/ReLU/MaxPool/ConvTranspose/Concat -> ramah ONNX & onnxruntime-web."""
    def __init__(self, in_ch=1, base=16, depth=4):
        super().__init__()
        ch = [base * 2 ** i for i in range(depth + 1)]
        self.enc = nn.ModuleList([DoubleConv(in_ch if i == 0 else ch[i - 1], ch[i]) for i in range(depth + 1)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(depth))])
        self.dec = nn.ModuleList([DoubleConv(ch[i] * 2, ch[i]) for i in reversed(range(depth))])
        self.head = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = []
        for i, enc in enumerate(self.enc):
            x = enc(x if i == 0 else F.max_pool2d(x, 2)); skips.append(x)
        x = skips.pop()
        for up, dec in zip(self.up, self.dec):
            x = dec(torch.cat([up(x), skips.pop()], 1))
        return self.head(x)   # logits

In [ ]:
# === 6b. Import model dari modul & cek ukuran ===
sys.path.insert(0, str(OUT / "src"))
from unet_lite import UNetLite
_m = UNetLite(CFG["in_channels"], CFG["base_ch"]).to(DEV)
n_params = sum(p.numel() for p in _m.parameters())
print(f"UNetLite params: {n_params/1e6:.3f} M  (~{n_params*4/1e6:.1f} MB FP32)")
with torch.no_grad(): print("output shape:", _m(torch.zeros(2, 1, 256, 256, device=DEV)).shape)
del _m

## 7. Training
AdamW + cosine, AMP, loss 0.5·BCE + 0.5·Dice, augmentasi dihedral + intensitas (hanya train), checkpoint terbaik berdasarkan Dice validation.

In [ ]:
# === 7. Training (GPU + AMP), BCE+Dice, early stopping pada Dice validation ===
CKPT = OUT / "checkpoints"; CKPT.mkdir(exist_ok=True)
seed_all(CFG["seed"])
model = UNetLite(CFG["in_channels"], CFG["base_ch"]).to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"], eta_min=CFG["lr"] * 0.01)
scaler = torch.amp.GradScaler("cuda")

def dice_loss(logits, y, eps=1.0):
    p = torch.sigmoid(logits).flatten(1); y = y.flatten(1)
    return (1 - (2 * (p * y).sum(1) + eps) / (p.sum(1) + y.sum(1) + eps)).mean()
def loss_fn(logits, y):
    w = CFG["bce_weight"]; return w * F.binary_cross_entropy_with_logits(logits, y) + (1 - w) * dice_loss(logits, y)

def augment(x, y):
    """Augmentasi hanya untuk TRAIN. Geometrik (8 transformasi dihedral) identik untuk citra & mask; intensitas hanya citra."""
    k = np.random.randint(4, size=len(x)); fl = np.random.rand(len(x)) < 0.5
    xs, ys = [], []
    for i in range(len(x)):
        xi, yi = torch.rot90(x[i], int(k[i]), (-2, -1)), torch.rot90(y[i], int(k[i]), (-2, -1))
        if fl[i]: xi, yi = xi.flip(-1), yi.flip(-1)
        xs.append(xi); ys.append(yi)
    x, y = torch.stack(xs), torch.stack(ys)
    x = x * torch.empty(len(x), 1, 1, 1, device=DEV).uniform_(0.9, 1.1) + torch.empty(len(x), 1, 1, 1, device=DEV).uniform_(-0.1, 0.1)
    return x, y

@torch.no_grad()
def predict_prob(m, gX, bs=64):
    m.eval(); out = []
    for s in range(0, len(gX), bs):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.sigmoid(m(prep(gX[s:s+bs])).float()).squeeze(1))
    return torch.cat(out)

hist, best, bad, t_start = [], -1, 0, time.time()
N, bs = len(gXtr), CFG["batch_size"]
for ep in range(1, CFG["epochs"] + 1):
    model.train(); t0 = time.time(); perm = torch.randperm(N, device=DEV); tl = 0.0
    for s in range(0, N - bs + 1, bs):
        idx = perm[s:s+bs]; x, y = augment(prep(gXtr[idx]), gYtr[idx].float().unsqueeze(1))
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(x).float(), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tl += loss.item() * bs
    sched.step()
    pv = predict_prob(model, gXva)
    with torch.no_grad():
        vl = sum(loss_fn(torch.logit(pv[s:s+64].clamp(1e-6, 1 - 1e-6)).unsqueeze(1), gYva[s:s+64].float().unsqueeze(1)).item() * len(pv[s:s+64]) for s in range(0, len(pv), 64)) / len(pv)
    vm, _ = seg_metrics(pv >= 0.5, gYva)
    rec = dict(epoch=ep, train_loss=tl / (N // bs * bs), val_loss=vl, val_dice=vm["dice_global"], val_iou=vm["iou_global"],
               val_dice_macro=vm["dice_macro"], lr=opt.param_groups[0]["lr"], sec=time.time() - t0)
    hist.append(rec)
    flag = ""
    if rec["val_dice"] > best:
        best, bad, flag = rec["val_dice"], 0, " *best*"
        torch.save(dict(state_dict=model.state_dict(), cfg=CFG, mean=MEAN, std=STD, epoch=ep, val_dice=best), CKPT / "unet_lite_best.pt")
    else:
        bad += 1
    print(f"ep {ep:02d} | train {rec['train_loss']:.4f} | val {vl:.4f} | vDice {rec['val_dice']:.4f} vIoU {rec['val_iou']:.4f} | {rec['sec']:.1f}s{flag}", flush=True)
    if bad >= CFG["patience"]:
        print(f"Early stopping di epoch {ep} (tidak membaik {CFG['patience']} epoch)"); break
pd.DataFrame(hist).to_csv(OUT / "training_log.csv", index=False)
json.dump(dict(cfg=CFG, versions=VERSIONS, mean=MEAN, std=STD, best_val_dice=best, total_train_sec=time.time() - t_start,
               gpu=torch.cuda.get_device_name(0)), open(OUT / "train_config.json", "w"), indent=1)
print(f"Selesai. best val Dice={best:.4f} | total {(time.time()-t_start)/60:.1f} menit")

## 8. Kurva training & threshold (validation)

In [ ]:
# === 8. Kurva training & pemilihan threshold di VALIDATION (bukan test) ===
H = pd.DataFrame(hist)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(H.epoch, H.train_loss, label="train"); ax[0].plot(H.epoch, H.val_loss, label="val"); ax[0].set_title("Loss (0.5 BCE + 0.5 Dice)"); ax[0].legend()
ax[1].plot(H.epoch, H.val_dice, label="val Dice (global)"); ax[1].plot(H.epoch, H.val_iou, label="val IoU (global)"); ax[1].plot(H.epoch, H.val_dice_macro, label="val Dice (macro)")
ax[1].set_title("Validation @ threshold 0.5"); ax[1].legend()
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=110); plt.show()

ck = torch.load(CKPT / "unet_lite_best.pt", map_location=DEV)
model.load_state_dict(ck["state_dict"]); model.eval()
print(f"Checkpoint terbaik: epoch {ck['epoch']} | val Dice@0.5 = {ck['val_dice']:.4f}")
pv = predict_prob(model, gXva)
ths = np.round(np.arange(0.20, 0.81, 0.05), 2)
tv = pd.DataFrame([dict(threshold=t, **{k: v for k, v in seg_metrics(pv >= t, gYva)[0].items()
     if k in ("dice_global", "iou_global", "precision_global", "recall_global", "dice_macro")}) for t in ths])
THRESH = float(tv.threshold[tv.dice_global.idxmax()])
tv.to_csv(OUT / "val_threshold_sweep.csv", index=False)
print(tv.round(4).to_string(index=False)); print("THRESHOLD dipilih di validation:", THRESH)

## 9. Evaluasi final test (sekali)
Agregasi global (piksel) dan macro (per citra). Juga sensitivitas tanpa near-duplicate dan uji lintas sensor PALSAR (terpisah).

In [ ]:
# === 9. Evaluasi FINAL pada test set Sentinel (sekali; model & threshold dipilih dari validation) ===
EVAL = OUT / "eval"; EVAL.mkdir(exist_ok=True)
pt = predict_prob(model, gXte)
m_test, dice_img = seg_metrics(pt >= THRESH, gYte); m_test["pr_auc_pixel"] = pr_auc(pt, gYte)
m_val, _ = seg_metrics(pv >= THRESH, gYva); m_val["pr_auc_pixel"] = pr_auc(pv, gYva)
# Sensitivitas: buang citra test yang near-duplicate (aHash dihedral <= TH) dengan train
keep = torch.tensor((test_df.nn_dist_to_train > TH).values, device=DEV)
m_test_nd, _ = seg_metrics(pt[keep] >= THRESH, gYte[keep]); m_test_nd["pr_auc_pixel"] = pr_auc(pt[keep], gYte[keep])
RESULTS = {"unet_lite/val": m_val, "unet_lite/test": m_test, "unet_lite/test_excl_neardup": m_test_nd, **BASE}

# Generalisasi lintas sensor (dilaporkan TERPISAH; model tidak pernah melihat PALSAR)
pal_te = df[(df.sensor == "palsar") & (df.split == "test")].copy(); pal_te["id"] = pal_te.stem.astype(int); pal_te = pal_te.sort_values("id")
Xp, Yp = load_set(pal_te); gXp, gYp = to_gpu(Xp), to_gpu(Yp)
pp = predict_prob(model, gXp); m_pal, _ = seg_metrics(pp >= THRESH, gYp); m_pal["pr_auc_pixel"] = pr_auc(pp, gYp)
RESULTS["unet_lite(sentinel-trained)/palsar_test_cross_sensor"] = m_pal
RESULTS[f"dark_threshold_t{T_DARK}/palsar_test_cross_sensor"], _ = seg_metrics(dark_spot(gXp, T_DARK), gYp)

res = pd.DataFrame(RESULTS).T
res.to_csv(EVAL / "metrics_summary.csv")
json.dump(dict(threshold=THRESH, aggregation=dict(pixel_global="TP/FP/FN/TN dijumlah atas semua piksel semua citra, lalu metrik dihitung",
    macro="metrik per citra lalu dirata-rata; GT & prediksi kosong = 1", macro_fg_only="macro hanya pada citra dengan GT berisi minyak",
    pr_auc="average precision piksel dari histogram 1000 bin skor probabilitas"), results=RESULTS),
    open(EVAL / "metrics.json", "w"), indent=1, default=float)
pd.DataFrame(dict(stem=test_df.stem, dice=dice_img, fg_frac_gt=test_df.fg_frac, pred_frac=(pt >= THRESH).float().mean((1, 2)).cpu().numpy(),
    nn_dist_to_train=test_df.nn_dist_to_train)).to_csv(EVAL / "per_image_test_dice.csv", index=False)
cols = ["n_images", "dice_global", "iou_global", "precision_global", "recall_global", "pr_auc_pixel", "dice_macro", "iou_macro", "dice_macro_fg_only"]
with pd.option_context("display.width", 250): print(res.reindex(columns=cols).round(4).to_string())
print("\nConfusion matrix piksel (test Sentinel):", {k: m_test[k] for k in ["TP", "FP", "FN", "TN"]})

fig, ax = plt.subplots(figsize=(4, 3.4))
cm = np.array([[m_test["TN"], m_test["FP"]], [m_test["FN"], m_test["TP"]]]); cmn = cm / cm.sum(1, keepdims=True)
ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2): ax.text(j, i, f"{cm[i, j]/1e6:.2f}M\n({cmn[i, j]:.1%})", ha="center", va="center", color="white" if cmn[i, j] > .5 else "black")
ax.set_xticks([0, 1], ["pred: laut", "pred: minyak"]); ax.set_yticks([0, 1], ["GT: laut", "GT: minyak"]); ax.set_title(f"Confusion matrix piksel (test, thr={THRESH})", fontsize=9)
plt.tight_layout(); plt.savefig(EVAL / "confusion_matrix_test.png", dpi=120); plt.show()

## 10. Analisis error visual

In [ ]:
# === 10. Visualisasi test: kasus terbaik, median, terburuk (+ GT kosong) — bukan hanya contoh terbaik ===
pred_te = (pt >= THRESH).cpu().numpy(); gt_te = Yte.astype(bool)
per = pd.read_csv(EVAL / "per_image_test_dice.csv")
has_fg = per.fg_frac_gt > 0
order = per[has_fg].sort_values("dice")
partial = per[(per.fg_frac_gt > 0.05) & (per.fg_frac_gt < 0.95)].sort_values("dice")   # "terbaik" dari mask parsial (bukan citra 100% minyak)
pick = [("terburuk", i) for i in order.index[:3]] + [("median", i) for i in order.index[len(order)//2 - 1: len(order)//2 + 2]] \
     + [("terbaik", i) for i in partial.index[-3:]] + [("GT kosong", i) for i in per[~has_fg].sort_values("pred_frac", ascending=False).index[:2]]
def err_map(p, g):
    e = np.zeros(p.shape + (3,), np.float32); e[p & g] = (0.1, 0.8, 0.2); e[p & ~g] = (1.0, 0.2, 0.1); e[~p & g] = (0.2, 0.4, 1.0); return e
fig, axes = plt.subplots(len(pick), 4, figsize=(10, 2.5 * len(pick)))
for r, (tag, i) in enumerate(pick):
    img = Xte[i]
    for c, (x, t) in enumerate([(img, "citra"), (gt_te[i].astype(np.uint8) * 255, "ground truth"), (overlay(img, pred_te[i] * 255), "prediksi"), (err_map(pred_te[i], gt_te[i]), "error: TP hijau, FP merah, FN biru")]):
        axes[r, c].imshow(x, cmap="gray", vmin=0, vmax=255) if x.ndim == 2 else axes[r, c].imshow(x); axes[r, c].axis("off")
        axes[r, c].set_title((f"[{tag}] id={per.stem[i]} Dice={per.dice[i]:.3f}  " if c == 0 else "") + t, fontsize=8)
plt.tight_layout(); plt.savefig(EVAL / "test_predictions_best_median_worst.png", dpi=100); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(per.dice[has_fg], bins=40); ax[0].set_title("Distribusi Dice per citra (test, GT berisi minyak)"); ax[0].set_xlabel("Dice")
ax[1].scatter(per.fg_frac_gt, per.dice, s=6, alpha=.5); ax[1].set_xlabel("proporsi foreground GT"); ax[1].set_ylabel("Dice"); ax[1].set_title("Dice vs luas tumpahan")
plt.tight_layout(); plt.savefig(EVAL / "test_dice_distribution.png", dpi=110); plt.show()
print("Dice per-citra (GT berisi minyak): median=%.3f | p10=%.3f | p90=%.3f | Dice<0.5: %d dari %d" % (
    per.dice[has_fg].median(), per.dice[has_fg].quantile(.1), per.dice[has_fg].quantile(.9), (per.dice[has_fg] < .5).sum(), has_fg.sum()))
print("Citra GT 100%% minyak: %d | Dice-nya: %s" % ((per.fg_frac_gt == 1).sum(), per.dice[per.fg_frac_gt == 1].round(3).tolist()))
print("GT kosong: %d citra | rata-rata piksel diprediksi minyak pada citra tsb = %.2f%%" % ((~has_fg).sum(), 100 * per.pred_frac[~has_fg].mean()))

## 11. Edge AI: ONNX + kuantisasi
Dijalankan sebagai subprocess (`src/export_onnx.py`) karena konflik versi protobuf di kernel. Varian web = terkecil dengan penurunan Dice validation ≤ 0.005.

In [ ]:
%%writefile /content/outputs/src/export_onnx.py
# === 11. Ekspor ONNX + FP16 + INT8 (QDQ), verifikasi akurasi & latency, siapkan artefak web ===
# Dijalankan sebagai subprocess agar paket onnx/protobuf terisolasi dari kernel notebook (kagglehub memuat protobuf lama).
import json, os, shutil, sys, time, platform
from pathlib import Path
import numpy as np, torch
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
from onnxconverter_common import float16
sys.path.insert(0, str(Path(__file__).parent))
from unet_lite import UNetLite

OUT = Path(sys.argv[1] if len(sys.argv) > 1 else "/content/outputs")
C = np.load(OUT / "cache" / "eval_arrays.npz")
Xcal, Xva, Yva, Xte, Yte, pred_te_torch = (C[k] for k in ["Xcal", "Xva", "Yva", "Xte", "Yte", "pred_te"])
ck = torch.load(OUT / "checkpoints" / "unet_lite_best.pt", map_location="cpu", weights_only=False)
cfg, MEAN, STD = ck["cfg"], ck["mean"], ck["std"]; S = cfg["img_size"]
THRESH = json.load(open(OUT / "eval" / "metrics.json"))["threshold"]
m = UNetLite(cfg["in_channels"], cfg["base_ch"]); m.load_state_dict(ck["state_dict"]); m.eval()
norm = lambda X: ((X[:, None].astype(np.float32) / 255.0) - MEAN) / STD

D = OUT / "onnx"; D.mkdir(exist_ok=True)
P = {"fp32": D / "unet_lite_fp32.onnx", "fp16": D / "unet_lite_fp16.onnx", "int8": D / "unet_lite_int8_qdq.onnx"}
dummy = torch.zeros(1, 1, S, S)
try:
    torch.onnx.export(m, (dummy,), str(P["fp32"]), input_names=["image"], output_names=["logits"], opset_version=17, dynamo=False)
    EXPORTER = "torch.onnx.export (TorchScript), opset 17"
except Exception as e:
    print("Exporter TorchScript gagal -> dynamo:", repr(e)[:300])
    torch.onnx.export(m, (dummy,), str(P["fp32"]), input_names=["image"], output_names=["logits"], opset_version=18, dynamo=True, external_data=False)
    EXPORTER = "torch.onnx.export (dynamo), opset 18"
g = onnx.load(str(P["fp32"])); onnx.checker.check_model(g)
print("Exporter:", EXPORTER, "| ops:", sorted({n.op_type for n in g.graph.node}), flush=True)

# FP16: bobot & komputasi FP16, I/O tetap float32 (kontrak frontend sama)
onnx.save(float16.convert_float_to_float16(g, keep_io_types=True), str(P["fp16"]))

# INT8 statis QDQ per-channel; kalibrasi 200 citra dari subset TRAIN
class Calib(CalibrationDataReader):
    def __init__(self, X): self.it = iter([{"image": norm(X[i:i + 1])} for i in range(len(X))])
    def get_next(self): return next(self.it, None)
pre = D / "_fp32_pre.onnx"; quant_pre_process(str(P["fp32"]), str(pre))
quantize_static(str(pre), str(P["int8"]), Calib(Xcal), quant_format=QuantFormat.QDQ, per_channel=True,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
pre.unlink()

def sess(path, threads=0):
    so = ort.SessionOptions(); so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL; so.intra_op_num_threads = threads
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
def probs(s, X):
    out = np.empty(X.shape, np.float32)
    for i in range(len(X)): out[i] = s.run(None, {"image": norm(X[i:i + 1])})[0][0, 0]
    return 1 / (1 + np.exp(-out))
def gm(pred, gt):
    p, t = pred.astype(bool), gt.astype(bool); tp = np.sum(p & t); fp = np.sum(p & ~t); fn = np.sum(~p & t)
    return dict(dice=float(2 * tp / (2 * tp + fp + fn)), iou=float(tp / (tp + fp + fn)), precision=float(tp / (tp + fp)), recall=float(tp / (tp + fn)))
def lat(s, n=50, warm=5):
    x = norm(Xte[:1])
    for _ in range(warm): s.run(None, {"image": x})
    ts = []
    for _ in range(n): t0 = time.perf_counter(); s.run(None, {"image": x}); ts.append((time.perf_counter() - t0) * 1000)
    return dict(median=round(float(np.median(ts)), 2), p90=round(float(np.percentile(ts, 90)), 2))

with torch.no_grad(): ref = torch.sigmoid(m(torch.from_numpy(norm(Xte[:8])))).numpy()[:, 0]
cpu = next((l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
R, PT = {}, {}
for name, path in P.items():
    s_all, s_1 = sess(path), sess(path, 1)
    t0 = time.time(); pv = probs(s_all, Xva); PT[name] = pt = probs(s_all, Xte)
    mv, mt = gm(pv >= THRESH, Yva), gm(pt >= THRESH, Yte)
    R[name] = dict(file=path.name, size_mb=round(os.path.getsize(path) / 1e6, 3),
        val_dice=round(mv["dice"], 4), val_iou=round(mv["iou"], 4), test_dice=round(mt["dice"], 4), test_iou=round(mt["iou"], 4),
        test_precision=round(mt["precision"], 4), test_recall=round(mt["recall"], 4),
        max_abs_prob_diff_vs_torch_fp32_cpu=float(np.abs(pt[:8] - ref).max()),
        test_pixel_agreement_vs_torch_gpu_mask=round(float(((pt >= THRESH) == pred_te_torch).mean()), 5),
        cpu_latency_ms_all_threads=lat(s_all), cpu_latency_ms_1_thread=lat(s_1), eval_sec=round(time.time() - t0, 1))
    print(name, R[name], flush=True)

if torch.cuda.is_available():
    mg = m.cuda(); xg = torch.from_numpy(norm(Xte[:1])).cuda()
    for tag, amp in [("torch_gpu_fp32", False), ("torch_gpu_amp_fp16", True)]:
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            for _ in range(10): mg(xg)
            torch.cuda.synchronize(); ts = []
            for _ in range(100): t0 = time.perf_counter(); mg(xg); torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1000)
        R[tag] = dict(latency_ms=dict(median=round(float(np.median(ts)), 2), p90=round(float(np.percentile(ts, 90)), 2)), device=torch.cuda.get_device_name(0))

# Pilih varian web berdasarkan VALIDATION: terkecil yang penurunan val Dice-nya <= 0.005 dari FP32
ok = [k for k in P if R["fp32"]["val_dice"] - R[k]["val_dice"] <= 0.005]
WEB = min(ok, key=lambda k: R[k]["size_mb"])
ENV = dict(onnxruntime=ort.__version__, onnx=onnx.__version__, torch=torch.__version__, exporter=EXPORTER, cpu=cpu, n_cpu=os.cpu_count(),
    latency_protocol="batch=1, input float32 1x1x256x256; ORT CPUExecutionProvider; 5 warm-up + 50 run; median & p90 wall-clock sess.run(); tanpa pre/post-processing. GPU: 10 warm-up + 100 run dengan cuda.synchronize.",
    web_variant_rule="varian terkecil dengan penurunan Dice VALIDATION <= 0.005 terhadap FP32", web_variant=WEB, threshold=THRESH)
json.dump(dict(env=ENV, results=R), open(D / "edge_benchmark.json", "w"), indent=1)
print("ENV:", ENV, flush=True)
np.save(OUT / "cache" / "web_variant_test_probs.npy", PT[WEB].astype(np.float16))

In [ ]:
# === 11b. Simpan array evaluasi ke cache lalu jalankan skrip ekspor ONNX (subprocess) ===
CACHE = OUT / "cache"; CACHE.mkdir(exist_ok=True)
cal_idx = np.random.default_rng(CFG["seed"]).choice(len(Xtr), 200, replace=False)   # kalibrasi INT8 dari TRAIN saja
np.savez_compressed(CACHE / "eval_arrays.npz", Xcal=Xtr[cal_idx], Xva=Xva, Yva=Yva, Xte=Xte, Yte=Yte, pred_te=pred_te)
json.dump(dict(json.load(open(OUT / "eval" / "metrics.json")), threshold=THRESH), open(OUT / "eval" / "metrics.json", "w"), indent=1, default=float)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common", "onnxscript"], check=True)
!python /content/outputs/src/export_onnx.py /content/outputs

## 12. Artefak web & unduh

In [ ]:
# === 12. Artefak web: model terpilih, metadata + kontrak I/O, sampel "Try Sample Data", zip output ===
import shutil, zipfile, datetime
EB = json.load(open(OUT / "onnx" / "edge_benchmark.json")); R = EB["results"]; WEB_VAR = EB["env"]["web_variant"]
print(pd.DataFrame({k: v for k, v in R.items() if k in ("fp32", "fp16", "int8")}).T[["size_mb", "val_dice", "test_dice", "test_iou", "test_precision", "test_recall", "cpu_latency_ms_all_threads", "cpu_latency_ms_1_thread"]].to_string())
print("GPU:", {k: v for k, v in R.items() if k.startswith("torch")}); print("Varian web (dipilih via validation):", WEB_VAR)

WEBD = OUT / "web"; (WEBD / "model").mkdir(parents=True, exist_ok=True); (WEBD / "samples").mkdir(exist_ok=True)
src_model = OUT / "onnx" / R[WEB_VAR]["file"]; shutil.copy(src_model, WEBD / "model" / src_model.name)
pw = np.load(CACHE / "web_variant_test_probs.npy").astype(np.float32) >= THRESH
def dice_np(p, g):
    tp = (p & g).sum(); d = 2 * tp + (p & ~g).sum() + (~p & g).sum(); return 1.0 if d == 0 else float(2 * tp / d)
dice_web = np.array([dice_np(pw[i], Yte[i].astype(bool)) for i in range(len(Yte))])

# Sampel representatif (bukan hanya terbaik): Dice ~ persentil 90/50/15 pada GT parsial 5-60%, + 1 citra tanpa minyak (median prediksi)
cand = np.where((test_df.fg_frac > 0.05) & (test_df.fg_frac < 0.6))[0]; chosen = []
for q, note in [(0.9, "kasus mudah (Dice ~ persentil 90)"), (0.5, "kasus tipikal (Dice ~ median)"), (0.15, "kasus sulit (Dice ~ persentil 15)")]:
    i = cand[np.argmin(np.abs(dice_web[cand] - np.quantile(dice_web[cand], q)))]; chosen.append((i, note))
emp = np.where(test_df.fg_frac == 0)[0]; fr = pw[emp].mean((1, 2)); chosen.append((emp[np.argsort(fr)[len(fr) // 2]], "tanpa tumpahan (GT kosong)"))
samples = []
for k, (i, note) in enumerate(chosen, 1):
    stem = test_df.stem[i]; im_name, mk_name = f"sample{k}_sentinel_test_{stem}.png", f"sample{k}_sentinel_test_{stem}_mask.png"
    shutil.copy(test_df.img_path[i], WEBD / "samples" / im_name); shutil.copy(test_df.lbl_path[i], WEBD / "samples" / mk_name)
    samples.append(dict(image=im_name, mask=mk_name, source=f"SOS dataset: test/sentinel/image/{stem}.png", note=note,
                        gt_fg_frac=round(float(test_df.fg_frac[i]), 4), pred_fg_frac=round(float(pw[i].mean()), 4), dice_python=round(dice_web[i], 4)))
json.dump(samples, open(WEBD / "samples" / "samples.json", "w"), indent=1)
DATASET = dict(name="Deep-SAR SOS Oil Spill Detection Dataset (Synthetic Aperture Radar Oil Spill Segmentation Dataset, SOS)",
    kaggle_handle="bitsandlayers/sar-oil-spill-segmentation-dataset-sos", url="https://www.kaggle.com/datasets/bitsandlayers/sar-oil-spill-segmentation-dataset-sos",
    version="1 (kagglehub)", owner="BitsandLayers (Kaggle)", license="CC BY 4.0 (Attribution 4.0 International) - sebagaimana tertera di halaman Kaggle",
    accessed="2026-09-24", regions="Gulf of Mexico (ALOS PALSAR), Persian Gulf (Sentinel-1A)")
(WEBD / "samples" / "ATTRIBUTION.txt").write_text(
    "Sample images & masks are unmodified files from the " + DATASET["name"] + "\n" + DATASET["url"] + "\nOwner: " + DATASET["owner"] +
    "\nLicense: " + DATASET["license"] + "\nAccessed: " + DATASET["accessed"] + "\nFiles: " + ", ".join(s["source"] for s in samples) + "\n")
META = dict(model_name="UNetLite (U-Net ringan, base 16, 1.94M parameter)", model_file=src_model.name, variant=WEB_VAR,
    model_size_mb=R[WEB_VAR]["size_mb"], threshold=THRESH, labels={"0": "background (laut / bukan minyak)", "1": "oil spill"},
    preprocessing=dict(PREPROC, mean=MEAN, std=STD),
    io=dict(input=dict(name="image", dtype="float32", shape=[1, 1, 256, 256], layout="NCHW (1 channel grayscale)"),
            output=dict(name="logits", dtype="float32", shape=[1, 1, 256, 256], postprocess="prob = sigmoid(logits); mask = prob >= threshold")),
    metrics_test_sentinel=dict(pytorch=dict((k, RESULTS["unet_lite/test"][k]) for k in ["dice_global", "iou_global", "precision_global", "recall_global", "pr_auc_pixel", "dice_macro", "iou_macro"]),
                               onnx_web_variant={k: R[WEB_VAR][k] for k in ["test_dice", "test_iou", "test_precision", "test_recall"]}),
    latency_note="Latency CPU di atas diukur di Colab (ORT CPU); latency browser tergantung perangkat & diukur di halaman web.",
    cpu_latency_ms_colab=R[WEB_VAR]["cpu_latency_ms_all_threads"], training=dict(sensor=CFG["sensor"], n_train=len(tr_df), n_val=len(va_df), n_test=len(test_df), best_epoch=ck["epoch"], seed=CFG["seed"]),
    dataset=DATASET, created_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))
json.dump(META, open(WEBD / "model" / "model_metadata.json", "w"), indent=1, default=float)
print(json.dumps(samples, indent=1))

ZIP = Path("/content/oil_spill_sar_outputs.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and not {"cache", "__pycache__"} & set(p.parts): z.write(p, p.relative_to(OUT))
print(f"ZIP: {ZIP} ({ZIP.stat().st_size/1e6:.1f} MB)")
for p in sorted(OUT.rglob("*")):
    if p.is_file() and not {"cache", "__pycache__"} & set(p.parts): print(f"{p.stat().st_size/1e6:8.3f} MB  {p.relative_to(OUT)}")

In [ ]:
# === 13. Unduh output ke komputer lokal (simpan juga ke Drive bila perlu agar tidak hilang saat runtime reset) ===
from google.colab import files
files.download(str(ZIP))